In [1]:
import sys
from pathlib import Path
import importlib

sys.path.append(str(Path("src").resolve()))

import model_selfdistill
importlib.reload(model_selfdistill)

from model_selfdistill import UNetSelfDistill

model_sd = UNetSelfDistill(num_classes=19)
print(model_sd)

UNetSelfDistill(
  (inc): DoubleConv(
    (block): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (down1): Down(
    (mpconv): Sequential(
      (0): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (1): DoubleConv(
        (block): Sequential(
          (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        

In [2]:
total_params_sd = sum(p.numel() for p in model_sd.parameters())
trainable_params_sd = sum(p.numel() for p in model_sd.parameters() if p.requires_grad)

print(f"Total parameters     : {total_params_sd:,}")
print(f"Trainable parameters : {trainable_params_sd:,}")

Total parameters     : 31,046,137
Trainable parameters : 31,046,137


In [3]:
import torch

x = torch.randn(1, 3, 256, 512)

with torch.no_grad():
    outputs = model_sd(x)

print("Main output :", outputs["main"].shape)
print("Aux1 output :", outputs["aux1"].shape)
print("Aux2 output :", outputs["aux2"].shape)

Main output : torch.Size([1, 19, 256, 512])
Aux1 output : torch.Size([1, 19, 256, 512])
Aux2 output : torch.Size([1, 19, 256, 512])


In [4]:
activations = {}

def get_hook(name):
    def hook(module, inp, out):
        activations[name] = out.shape
    return hook

hooks = []
hooks.append(model_sd.up2.register_forward_hook(get_hook("up2_out")))
hooks.append(model_sd.up3.register_forward_hook(get_hook("up3_out")))
hooks.append(model_sd.up4.register_forward_hook(get_hook("up4_out")))
hooks.append(model_sd.aux_head2.register_forward_hook(get_hook("aux_head2_out")))
hooks.append(model_sd.aux_head1.register_forward_hook(get_hook("aux_head1_out")))
hooks.append(model_sd.main_head.register_forward_hook(get_hook("main_head_out")))

with torch.no_grad():
    _ = model_sd(torch.randn(1, 3, 256, 512))

for k, v in activations.items():
    print(f"{k}: {v}")

for h in hooks:
    h.remove()

up2_out: torch.Size([1, 256, 64, 128])
up3_out: torch.Size([1, 128, 128, 256])
up4_out: torch.Size([1, 64, 256, 512])
main_head_out: torch.Size([1, 19, 256, 512])
aux_head2_out: torch.Size([1, 19, 64, 128])
aux_head1_out: torch.Size([1, 19, 128, 256])


In [5]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_sd = UNetSelfDistill(num_classes=19).to(device)
checkpoint_sd = torch.load("checkpoints/unet_selfdistill_best.pth", map_location=device)
model_sd.load_state_dict(checkpoint_sd["model_state_dict"])
model_sd.eval()

print("Best self-distilled mIoU:", checkpoint_sd.get("best_miou", "N/A"))

Best self-distilled mIoU: 0.08039171248674393


In [6]:
import sys
from pathlib import Path
sys.path.append(str(Path("src").resolve()))

from dataset import CityscapesSegmentation, JointTransform
from torch.utils.data import DataLoader

val_dataset = CityscapesSegmentation(
    root="data/cityscapes",
    split="val",
    transform=JointTransform(image_size=(256, 512), train=False)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

print("Validation set size:", len(val_dataset))

Validation set size: 500


In [7]:
import sys
from pathlib import Path
sys.path.append(str(Path("src").resolve()))

from metrics import compute_confusion_matrix, compute_iou_from_confmat, pixel_accuracy

total_confmat_sd = torch.zeros((19, 19), dtype=torch.int64, device=device)
acc_list_sd = []

model_sd.eval()
with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model_sd(images)
        preds = torch.argmax(outputs["main"], dim=1)

        total_confmat_sd += compute_confusion_matrix(preds, masks, num_classes=19, ignore_index=255)
        acc_list_sd.append(pixel_accuracy(preds, masks, ignore_index=255))

iou_per_class_sd, miou_sd = compute_iou_from_confmat(total_confmat_sd)
avg_acc_sd = sum(acc_list_sd) / len(acc_list_sd)

print("Self-Distilled Validation Pixel Accuracy:", avg_acc_sd)
print("Self-Distilled Validation mIoU:", miou_sd)

Self-Distilled Validation Pixel Accuracy: 0.481572742096008
Self-Distilled Validation mIoU: 0.08039171248674393


In [ ]:
CITYSCAPES_CLASSES = [
    "road", "sidewalk", "building", "wall", "fence", "pole",
    "traffic_light", "traffic_sign", "vegetation", "terrain", "sky",
    "person", "rider", "car", "truck", "bus", "train",
    "motorcycle", "bicycle"
]

for cls, iou in zip(CITYSCAPES_CLASSES, iou_per_class_sd):
    print(f"{cls:15s}: {iou:.4f}")

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("src").resolve()))

from losses import SelfDistillationLoss

criterion_sd = SelfDistillationLoss(
    num_classes=19,
    ignore_index=255,
    temperature=2.0,   # ou 4.0 selon ta version
    main_weight=1.0,
    aux_weight=0.3,
    kd_weight=0.05,
    dice_weight=1.0
)

images, masks = next(iter(val_loader))
images = images.to(device)
masks = masks.to(device)

with torch.no_grad():
    outputs = model_sd(images)
    loss, stats = criterion_sd(outputs, masks)

print("Total loss:", loss.item())
for k, v in stats.items():
    print(f"{k}: {v}")